In [ ]:
import os
import base64
from anthropic import AnthropicFoundry
from anthropic.lib import files_from_dir
from dotenv import load_dotenv
load_dotenv(override=True)

#tenant_id = os.getenv("MAAS_TENANT_ID")
endpoint = os.getenv("ANTHROPIC_ENDPOINT")
api_key = os.getenv("ANTHROPIC_API_KEY")
deployment_name = os.getenv("CLAUDE_MODEL")

client = AnthropicFoundry(
    api_key=api_key,
    base_url=endpoint
)

### Create a skill

In [ ]:
skill = client.beta.skills.create(
    display_title="missing-dimension-check",
    files=files_from_dir(".claude/skills/missing-dimension-check"),
    betas=["skills-2025-10-02"],
)

print(f"Created skill: {skill.id}")
print(f"Latest version: {skill.latest_version}")

<h5> Display all skills </h5>

In [ ]:
skills = client.beta.skills.list(betas=["skills-2025-10-02"])

for skill in skills.data:
    print(f"{skill.id}: {skill.display_title} (source: {skill.source})")

print("\nList the custom Skills only")
custom_skills = client.beta.skills.list(source="custom", betas=["skills-2025-10-02"])

for skill in custom_skills.data:
    print(f"{skill.id}: {skill.display_title} (source: {skill.source})")


<h5>Retrieve a Skill</h5>

In [ ]:
skill_id = "<skill_id>"  # Replace with the actual skill ID you want to retrieve

skill = client.beta.skills.retrieve(
    skill_id=skill_id, betas=["skills-2025-10-02"]
)

print(f"Skill: {skill.display_title}")
print(f"Latest version: {skill.latest_version}")
print(f"Created: {skill.created_at}")

<h5> DELETING SKILLS </h5>

In [ ]:

# step1: Delete all versions of the Skill
versions = client.beta.skills.versions.list(
    skill_id=skill_id, betas=["skills-2025-10-02"]
)

for version in versions.data:
    client.beta.skills.versions.delete(
        skill_id=skill_id,
        version=version.version,
        betas=["skills-2025-10-02"],
    )

# step2: Delete the Skill itself
client.beta.skills.delete(
    skill_id=skill_id, betas=["skills-2025-10-02"]
)

<h5> DELETING FILES </h5>

In [ ]:
files = client.beta.files.list(limit=1000)

for file in files.data:
    print(f"Deleting file: {file.id} - {file.filename}")
    client.beta.files.delete(file.id)

### UPLOAD FILES

<h4>LISTING FILES</h4>

In [ ]:
# LISTING FILES
files = client.beta.files.list(limit=1000)
for file in files.data:
    print(f"{file.id}: {file.filename} ")
    if file.filename == "BaseStationDiagram_900dpi.png":
        uploaded = file
        print(f"\nFound target file: {uploaded.filename} with ID: {uploaded.id}")

print(f"\nTotal number of files: {len(files.data)}")

<h5> UPLOAD A FILE </h5>

In [ ]:
# UPLOADING FILES
uploaded1 = client.beta.files.upload(
    file=("sample01.png", open("data/sample01.png", "rb"), "image/png"),
)
print(f"Uploaded PNG: {uploaded1.id}")
uploaded2 = client.beta.files.upload(
    file=("check01.png", open("data/check01.png", "rb"), "image/png"),
)
print(f"Uploaded PNG: {uploaded2.id}")

<h5> UPLOAD A RESIZED FILE </h5>

In [ ]:
# Resize images before uploading to reduce file size with the long edge not exceeding 1568 pixels
from PIL import Image

def resize_long_edge(src_path, dst_path, max_long_edge=1568):
    with Image.open(src_path) as im:
        w, h = im.size
        scale = min(1.0, max_long_edge / max(w, h))  # Do not upscale
        if scale < 1.0:
            im = im.resize((round(w * scale), round(h * scale)), Image.LANCZOS)
        im.save(dst_path, format="PNG", optimize=True)
    before = os.path.getsize(src_path) / 1024
    after = os.path.getsize(dst_path) / 1024
    print(f"{os.path.basename(src_path)}: {w}x{h} ({before:.0f} KB) -> {im.size[0]}x{im.size[1]} ({after:.0f} KB)")
    return dst_path

resized1 = resize_long_edge("data/sample01.png", "data/sample01_resized.png")
resized2 = resize_long_edge("data/check01.png", "data/check01_resized.png")

uploaded1 = client.beta.files.upload(
    file=("sample01.png", open(resized1, "rb"), "image/png"),
)
print(f"Uploaded PNG: {uploaded1.id}")
uploaded2 = client.beta.files.upload(
    file=("check01.png", open(resized2, "rb"), "image/png"),
)
print(f"Uploaded PNG: {uploaded2.id}")

<h5>Retrieve uploaded files</h5>

In [ ]:
file_id1 = "file_011CcJNaJoDRi65J2aGDy5BQ"  # Replace with the actual file ID you want to retrieve
file_id2 = "file_011CcJNaDcRcd75P4iyh4Kij"  # Replace with the actual file ID you want to retrieve

uploaded1 = client.beta.files.retrieve_metadata(
    file_id=file_id1,
    betas=["files-api-2025-04-14"],
)
print(f"{uploaded1.id}: {uploaded1.filename}")
uploaded2 = client.beta.files.retrieve_metadata(
    file_id=file_id2,
    betas=["files-api-2025-04-14"],
)
print(f"{uploaded2.id}: {uploaded2.filename}")

### MULTI-TURN

In [ ]:
with open("user_prompt_template_01.txt", "r", encoding = 'utf-8') as f:
    user_prompt = f.read()

In [ ]:
# MULTI-TURN (checkpoint approach + streaming)
#   - Save a checkpoint to the container at each chunk and keep every request minimal
#     to avoid message-length / timeout (408) issues
#   - Streaming displays the response incrementally, so it never becomes a "silent wait" and is robust for long requests
#
# Policy:
#  - State is held in working/checkpoint.json inside the container, not in the "conversation history"
#  - Images are mounted with container_upload only on the first request (they stay in the container afterward, so no resend)
#  - Each request only sends a short instruction to "read the checkpoint and run just the next chunk"
#  - The save/resume protocol is described in the skill, so it can resume even with zero history
#  - Pass the response's container.id to the next request to reuse the same container (files + skill)

import time
import random
from anthropic import APIStatusError, APIConnectionError, RateLimitError

betas = ["files-api-2025-04-14", "code-execution-2025-08-25", "skills-2025-10-02"]
tools = [{"type": "code_execution_20250825", "name": "code_execution"}]
CHECKPOINT = "working/checkpoint.json"


def run_one_request(messages, container_ref, max_retries=8, base_delay=2.0, max_delay=60.0):
    """Send one request via streaming and auto-continue only during pause_turn until end_turn.
       - Prints the response text incrementally, so you can see that processing is progressing.
       - 408/429/529/5xx/connection errors are auto-retried with exponential backoff + jitter.
       - History growth is confined to the 'current chunk' only (it does not accumulate across chunks)."""
    while True:  # pause_turn auto-continue loop
        response = None
        for attempt in range(max_retries):
            try:
                with client.beta.messages.stream(
                    model=deployment_name,
                    max_tokens=16384,
                    betas=betas,
                    container=container_ref,
                    messages=messages,
                    tools=tools,
                ) as stream:
                    for text in stream.text_stream:
                        print(text, end="", flush=True)
                    response = stream.get_final_message()
                print()  # newline
                break  # success
            except (APIStatusError, RateLimitError, APIConnectionError) as e:
                status = getattr(e, "status_code", None)
                retryable = isinstance(e, (RateLimitError, APIConnectionError)) or (
                    status is not None and (status in (408, 409, 429, 529) or status >= 500)
                )
                if not retryable or attempt == max_retries - 1:
                    raise
                delay = min(max_delay, base_delay * (2 ** attempt)) + random.uniform(0, 1.0)
                print(f"\n⚠️ {type(e).__name__} (status={status}) — waiting {delay:.1f}s and retrying "
                      f"({attempt + 1}/{max_retries})")
                time.sleep(delay)

        if getattr(response, "container", None):
            container_ref = response.container.id

        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "pause_turn":
            print("⏸ pause_turn — auto-continue...")
            continue
        return response, container_ref


def is_done(response):
    return any(
        getattr(it, "type", None) == "text" and "[DONE]" in it.text
        for it in response.content
    )


# First request: prompt + image mount + checkpoint-operation instructions
first_instruction = (
    "Inspect the missing dimensions of the two attached drawings "
    "(baseline drawing sample01.png / drawing to check check01.png), following the skill's procedure.\n"
    "However, do not do all the steps at once; proceed in reasonable 'chunks' such as per Step.\n"
    f"At the end of each chunk, save progress to {CHECKPOINT} (including completed Steps, extracted dimensions, unprocessed regions, "
    "and paths to generated files), and at the end of the response write [CONTINUE] if incomplete or [DONE] if all steps are complete.\n"
    "The final deliverable is an image showing the missing dimensions as red circles on the drawing to check; record its path in the checkpoint and the final response.\n"
    "Please answer in English."
)

messages = [{"role": "user", "content": [
    {"type": "text", "text": user_prompt},
    {"type": "text", "text": "Attachment 1: mounting sample01.png (baseline drawing)."},
    {"type": "container_upload", "file_id": uploaded1.id},
    {"type": "text", "text": "Attachment 2: mounting check01.png (drawing to check)."},
    {"type": "container_upload", "file_id": uploaded2.id},
    {"type": "text", "text": first_instruction},
]}]

# Resume message for each chunk (does not use history; sends only the minimal instruction that references the checkpoint)
resume_instruction = (
    f"Read {CHECKPOINT} and execute only the next 'single chunk' from where you left off.\n"
    f"When done, update {CHECKPOINT} and at the end write [CONTINUE] if incomplete or [DONE] if all steps are complete."
)

# First time is a dict (new container + skill); afterward reuse the string container_id
container_ref = {"skills": [{"type": "custom", "skill_id": skill.id, "version": "latest"}]}
MAX_CHUNKS = 30

for chunk in range(MAX_CHUNKS):
    response, container_ref = run_one_request(messages, container_ref)
    print(f"\n[chunk {chunk + 1}/{MAX_CHUNKS}] stop_reason: {response.stop_reason}")

    if is_done(response):
        print("✅ Done")
        break

    # To the next chunk: discard history and resume with only the minimal message referencing the checkpoint
    messages = [{"role": "user", "content": [{"type": "text", "text": resume_instruction}]}]
else:
    print(f"⚠️ Reached MAX_CHUNKS ({MAX_CHUNKS})")


<h5> LISTING FILES </h5>

In [ ]:
# LISTING FILES
files = client.beta.files.list(limit=1000)
for file in files.data:
    print(f"{file.id}: {file.filename} ")
    if file.filename == "BaseStationDiagram_900dpi.png":
        uploaded = file
        print(f"\nFound target file: {uploaded.filename} with ID: {uploaded.id}")

print(f"\nTotal number of files: {len(files.data)}")

<h5> DOWNLOADING FILES </h5>

In [ ]:
# Download the created files (excluding the 2 original uploads) to ./out
# For files with the same name (e.g. two checkpoint.json), append a sequence number to avoid collisions
os.makedirs("out", exist_ok=True)

# The original files uploaded first (excluded from download)
original_ids = {uploaded1.id, uploaded2.id}

files = client.beta.files.list(limit=1000)
downloaded = []
for file in files.data:
    if file.id in original_ids:
        continue  # Skip the original uploads

    out_path = os.path.join("out", file.filename)
    base, ext = os.path.splitext(out_path)
    counter = 1
    while os.path.exists(out_path):
        out_path = f"{base}_{counter}{ext}"
        counter += 1

    file_content = client.beta.files.download(
        file_id=file.id, betas=["files-api-2025-04-14"]
    )
    file_content.write_to_file(out_path)
    downloaded.append(out_path)
    print(f"Downloaded: {file.id} -> {out_path}")

print(f"\nSaved {len(downloaded)} file(s) to ./out in total.")
